## Step 1 — Install Dependencies

In [4]:
# !pip install -q numpy==1.26.4
# !pip install -q torch==2.2.2+cu118 torchvision==0.17.2+cu118 torchaudio==2.2.2+cu118 --index-url https://download.pytorch.org/whl/cu118
# !pip install -q transformers==4.40.0 peft==0.10.0 trl==0.8.6 accelerate==0.29.3 bitsandbytes==0.43.1 sentencepiece protobuf
import torch
print(f"Torch version: {torch.__version__}")

Torch version: 2.2.2+cu118


In [5]:
!pip install numpy==1.26.4
!pip install torch==2.2.2+cu118 torchvision==0.17.2+cu118 torchaudio==2.2.2+cu118 --index-url https://download.pytorch.org/whl/cu118
!pip install transformers accelerate peft trl bitsandbytes==0.42.0
!pip install triton==2.1.0

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 131.2 MB/s  0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
fastai 2.8.7 requires torch<3,>=1.10, which is not installed.
fastai 2.8.7 requires torchvision>=0.11, which is not installed.
sentence-transformers 5.4.0 requires torch>=1.11.0, which is not installed.
sentence-transformers 5.4.0 requires transformers<6.0.0,>=4.41.0, which is not installed.
tobler 0.14.0 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
rasterio 1.5.0 requires numpy>=2, but you have numpy 1.26.4 which is incompatible.
shap 0.51.0 requires numpy>=2, but you have numpy 1.26.4 which is incompatible.
xarray-einstats 0.10.0 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
opencv-python-headless 4.13.0.92 requires numpy>=2; python_version >= "3.9", but you have numpy 1.26.4 which is inc

Looking in indexes: https://download.pytorch.org/whl/cu118
  Using cached https://download-r2.pytorch.org/whl/cu118/torch-2.2.2%2Bcu118-cp312-cp312-linux_x86_64.whl (819.1 MB)
  Using cached https://download-r2.pytorch.org/whl/cu118/torchvision-0.17.2%2Bcu118-cp312-cp312-linux_x86_64.whl (6.2 MB)
  Using cached https://download-r2.pytorch.org/whl/cu118/torchaudio-2.2.2%2Bcu118-cp312-cp312-linux_x86_64.whl (3.3 MB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3/3 [torchaudio]
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
sentence-transformers 5.4.0 requires transformers<6.0.0,>=4.41.0, which is not installed.


  Using cached transformers-5.5.4-py3-none-any.whl.metadata (32 kB)
  Using cached accelerate-1.13.0-py3-none-any.whl.metadata (19 kB)
  Using cached trl-1.2.0-py3-none-any.whl.metadata (11 kB)
  Using cached bitsandbytes-0.42.0-py3-none-any.whl.metadata (9.9 kB)
Using cached bitsandbytes-0.42.0-py3-none-any.whl (105.0 MB)
Using cached transformers-5.5.4-py3-none-any.whl (10.2 MB)
Using cached accelerate-1.13.0-py3-none-any.whl (383 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 680.7/680.7 kB 837.2 kB/s  0:00:00
Using cached trl-1.2.0-py3-none-any.whl (697 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5/5 [peft]
ERROR: Could not find a version that satisfies the requirement triton==2.1.0 (from versions: 2.2.0, 2.3.0, 2.3.1, 3.0.0, 3.1.0, 3.2.0, 3.3.0, 3.3.1, 3.4.0, 3.5.0, 3.5.1, 3.6.0)
ERROR: No matching distribution found for triton==2.1.0


## Step 2 — Check GPU

In [1]:
import torch

print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    total = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"VRAM: {total:.1f} GB")
else:
    print("⚠️  No GPU detected. Go to Runtime > Change runtime type > T4 GPU")

CUDA available: True
GPU: Tesla T4
VRAM: 15.6 GB


## Step 3 — Load & Format Dataset

In [1]:
import json
from datasets import Dataset

# ── Load JSON ──────────────────────────────────────────────────────────
DATA_PATH = "classifier_training_data_v2.json"  # upload this file to Colab

with open(DATA_PATH, "r") as f:
    raw_data = json.load(f)

print(f"Loaded {len(raw_data)} samples")

# ── Format into Mistral instruction template ───────────────────────────
# Mistral expects: <s>[INST] {instruction} [/INST] {output}</s>
def format_sample(sample):
    instruction = sample["instruction"]
    output      = sample["output"]
    # Include 'input' field if non-empty
    if sample.get("input", "").strip():
        user_msg = f"{instruction}\n\nContext: {sample['input']}"
    else:
        user_msg = instruction

    text = f"<s>[INST] {user_msg} [/INST] {output}</s>"
    return {"text": text}

formatted = [format_sample(s) for s in raw_data]
dataset   = Dataset.from_list(formatted)

# ── Train / Val split (90/10) ──────────────────────────────────────────
split   = dataset.train_test_split(test_size=0.1, seed=42)
train_ds = split["train"]
val_ds   = split["test"]

print(f"Train: {len(train_ds)} | Val: {len(val_ds)}")
print("\nSample formatted text:")
print(train_ds[0]["text"][:500])

Loaded 111 samples
Train: 99 | Val: 12

Sample formatted text:
<s>[INST] Build a QA bot for our product documentation [/INST] Questions:
1. What format is your documentation? (Markdown, HTML, PDF, Confluence)
2. How frequently does the documentation update?
3. Should it handle follow-up questions in conversation?
4. Fallback when answer not found?

JSON:
{
  "task": "question_answering",
  "domain": "software",
  "modality": "text",
  "dataset_provided": true,
  "classes": null
}</s>


## Step 4 — Load Model & Tokenizer (4-bit QLoRA for T4)

In [2]:
import torch
print(torch.__version__)
print(torch.version.cuda)
print(torch.cuda.is_available())

2.2.2+cu118
11.8
True


In [5]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, TaskType, prepare_model_for_kbit_training
from trl import SFTTrainer, SFTConfig

MODEL_ID = "mistralai/Mistral-7B-v0.3"

# ── 4bit config (T4-safe) ─────────────────────────────
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

# Load model with quantization
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.float16,
    trust_remote_code=True,
)

# Prepare for k-bit training
model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True)

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type=TaskType.CAUSAL_LM,
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

NameError: name 'torch' is not defined

In [8]:
!pip install bitsandbytes==0.42.0

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 105.0/105.0 MB 8.9 MB/s eta 0:00:00


In [ ]:
!pip uninstall -y bitsandbytes
!pip install bitsandbytes==0.43.1 --prefer-binary

# Force a runtime restart to clear the library cache
import os
os._exit(0)

Found existing installation: bitsandbytes 0.43.1
Uninstalling bitsandbytes-0.43.1:
  Successfully uninstalled bitsandbytes-0.43.1
  Using cached bitsandbytes-0.43.1-py3-none-manylinux_2_24_x86_64.whl.metadata (2.2 kB)
Using cached bitsandbytes-0.43.1-py3-none-manylinux_2_24_x86_64.whl (119.8 MB)


In [32]:
import torch

model = model.to(dtype=torch.float16)
model.config.torch_dtype = torch.float16

In [37]:
print(torch.__version__)

2.10.0+cu128


In [34]:
import torch

torch.cuda.amp.autocast(enabled=True, dtype=torch.float16)

/tmp/ipykernel_4812/1786599495.py:3: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  torch.cuda.amp.autocast(enabled=True, dtype=torch.float16)


In [35]:
import os
os.environ["TORCH_USE_CUDA_DSA"] = "0"
os.environ["ACCELERATE_DISABLE_RICH"] = "1"

In [10]:
# ── FIXED SFT CONFIG (torch 2.2 + TRL stable) ─────────
import torch

# Force model to FP16 to avoid the BFloat16 NotImplementedError on T4
model = model.to(torch.float16)
for param in model.parameters():
    if param.dtype == torch.bfloat16:
        param.data = param.data.to(torch.float16)

training_args = SFTConfig(
    output_dir="./classifier_adapter",
    max_length=1024,
    num_train_epochs=5,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=8,
    per_device_eval_batch_size=2,
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    warmup_steps=10,
    optim="paged_adamw_8bit",
    weight_decay=0.01,
    fp16=True,        # REQUIRED for T4
    bf16=False,       # MUST stay False (critical fix)
    logging_steps=5,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    report_to="none",
    dataloader_pin_memory=False,
)

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    processing_class=tokenizer,
)

print("🚀 Starting training (T4 + FP16 + 4bit QLoRA safe)...")
trainer.train()
print("✅ Done")

NameError: name 'model' is not defined

## Step 7 — Save Adapter Only

In [ ]:
# Save ONLY the LoRA adapter weights (~50-200MB, not the full 14GB model)
ADAPTER_SAVE_PATH = "./classifier_adapter_final"

model.save_pretrained(ADAPTER_SAVE_PATH)
tokenizer.save_pretrained(ADAPTER_SAVE_PATH)

print(f"✅ Adapter saved to {ADAPTER_SAVE_PATH}")

# Show saved files and size
import os
total_size = 0
for f in os.listdir(ADAPTER_SAVE_PATH):
    size = os.path.getsize(os.path.join(ADAPTER_SAVE_PATH, f))
    total_size += size
    print(f"  {f}: {size/1e6:.1f} MB")
print(f"Total adapter size: {total_size/1e6:.1f} MB")

## Step 8 — Download Adapter to Local Machine

In [ ]:
import shutil

# Zip the adapter folder for easy download
shutil.make_archive("classifier_adapter_final", "zip", ".", "classifier_adapter_final")
print("✅ Zipped adapter")

# Download
from google.colab import files
files.download("classifier_adapter_final.zip")
print("📦 Download started")

## Step 9 — Test the Fine-Tuned Adapter

In [ ]:
from transformers import pipeline
import torch

def run_classifier_agent(user_instruction: str, max_new_tokens: int = 300) -> str:
    """
    Run the fine-tuned classifier agent on a new instruction.
    Returns clarifying questions + structured JSON.
    """
    prompt = f"<s>[INST] {user_instruction} [/INST]"

    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=0.3,         # low temp for structured output
            do_sample=True,
            top_p=0.9,
            repetition_penalty=1.1,
            eos_token_id=tokenizer.eos_token_id,
            pad_token_id=tokenizer.eos_token_id,
        )

    # Decode only the new tokens (not the prompt)
    new_tokens = outputs[0][inputs["input_ids"].shape[1]:]
    return tokenizer.decode(new_tokens, skip_special_tokens=True)


# ── Test cases ────────────────────────────────────────────────────────
test_instructions = [
    "Build a system to detect fake reviews on Amazon",
    "I want to predict patient survival rates",
    "Make me an AI for my restaurant",
    "Classify satellite images of forests",
]

for instruction in test_instructions:
    print(f"\n{'='*60}")
    print(f"INPUT: {instruction}")
    print(f"{'='*60}")
    output = run_classifier_agent(instruction)
    print(output)

## Step 10 — (Optional) Push Adapter to HuggingFace Hub

In [ ]:
# Only run this if you want to push to HF Hub
# from huggingface_hub import login
# login(token="hf_YOUR_TOKEN_HERE")

# model.push_to_hub("your-username/forgefactory-classifier-adapter")
# tokenizer.push_to_hub("your-username/forgefactory-classifier-adapter")
# print("✅ Pushed to HuggingFace Hub")

print("Uncomment above lines and add your HF token to push adapter to Hub")

---
## How to load this adapter later on the MI300X

```python
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel
import torch

BASE_MODEL   = "mistralai/Mistral-7B-v0.3"
ADAPTER_PATH = "./classifier_adapter_final"  # or HF Hub path

# On MI300X — load in fp16 (no quantization needed, you have 192GB)
tokenizer = AutoTokenizer.from_pretrained(ADAPTER_PATH)
base      = AutoModelForCausalLM.from_pretrained(BASE_MODEL, torch_dtype=torch.float16, device_map="auto")
model     = PeftModel.from_pretrained(base, ADAPTER_PATH)
model.eval()

# Swap to searcher adapter later:
# model.load_adapter("./searcher_adapter")
```

---
## Notes
- Training 111 samples × 5 epochs on T4 ≈ **15–20 minutes**
- Adapter size ≈ **50–150MB** (not the full model)
- On MI300X: remove `BitsAndBytesConfig` and use `torch_dtype=torch.float16` — cleaner and faster
- To add more training data: just append more samples to the JSON and re-run